# Stage 3 v2: Group-relative PPO cho ViT5 (RL mạnh hơn bản cũ)

Notebook này thay cho `stage3_vit5_RLUsingPPO.ipynb` (giữ lại làm mốc so sánh). Lõi RL nằm trong package `summ_rl` của repo, có kiểm thử offline, nên notebook chỉ cấu hình và chạy.

**Vì sao bản cũ yếu và bản này sửa gì**

| Bản cũ | Bản v2 |
|---|---|
| Trọng số actor bf16 với LR 1e-6: bước cập nhật nhỏ hơn độ phân giải bf16 nên gần như không đổi | Trọng số fp32, chỉ forward dùng bf16 autocast, LR 1e-5 |
| 1 mẫu/prompt, critic (value head) học từ ~800 mẫu | 8 ứng viên/prompt, lợi thế so trong nhóm, không critic |
| ~147 bước cập nhật, mỗi batch dùng 1 lần | Mỗi batch lấy mẫu dùng cho 2 lần cập nhật clipped PPO, số bước cấu hình được |
| Reward chỉ luật số liệu/độ dài, không có tín hiệu nội dung | Reward gồm ROUGE-1/2/L so với tham chiếu, số liệu, độ dài, chống lặp, phạt hallucination số |
| Dropout bật khi lấy mẫu và khi cập nhật, tỉ số xác suất bị nhiễu | Tắt dropout, tỉ số = 1 ở lần cập nhật đầu |
| Chọn best trên 30 mẫu val | Chọn trên toàn bộ val, không thấp hơn SFT, loại nếu hallucination số tăng quá ngưỡng |

**Đầu vào**: checkpoint Stage 2 (SFT) và `health_training_v1/{train,val,test}/dataset.jsonl` (tạo bằng `scripts/build_training_data.py`, upload lên Drive).

**Lưu ý**: nhãn hiện là `extractive_pseudo` chưa duyệt. Điểm trên chúng chỉ để so sánh các cấu hình với nhau, không phải bằng chứng chất lượng tóm tắt thực tế.

In [ ]:
# Cell 1 - Cai dat va lay ma nguon summ_rl tu GitHub
import subprocess, sys, os
INSTALL_DEPS = False   # True neu runtime thieu package; sau khi cai xong hay restart runtime
if INSTALL_DEPS:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.46.3', 'tokenizers==0.20.3',
                           'sentencepiece==0.2.0', 'accelerate==1.1.1'])
    raise RuntimeError('Da cai xong. Restart runtime roi chay lai tu Cell 1 voi INSTALL_DEPS=False.')

REPO_URL = 'https://github.com/doanlamphuckhang/Text-Summary.git'
if not os.path.isdir('/content/Text-Summary'):
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, '/content/Text-Summary'])
sys.path.insert(0, '/content/Text-Summary')
import summ_rl
print('summ_rl OK')

In [ ]:
# Cell 2 - Cau hinh
import json, random
from pathlib import Path
import numpy as np
import torch

SFT_DIR   = '/content/drive/MyDrive/NLP_Final/Output stage 2 vit5 checkpoint/best_model'   # sua cho dung checkpoint Stage 2
DATA_DIR  = '/content/drive/MyDrive/NLP_Final/health_training_v1'                          # upload thu muc data/health_training_v1
OUT_DIR   = '/content/drive/MyDrive/NLP_Final/ViT5_stage_3_v2/run_01'                      # moi lan chay dung thu muc moi

bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
# Profile theo GPU: A100/L4 (bf16) dung profile lon; T4 (khong co bf16) dung profile nho va rat cham.
if bf16:
    PROFILE = dict(group_size=8, prompts_per_step=4, max_input_len=1024, max_new_tokens=320)
else:
    PROFILE = dict(group_size=4, prompts_per_step=2, max_input_len=768, max_new_tokens=256)
STEPS, EVAL_EVERY = 150, 25
RESUME = False         # True: tiep tuc run bi ngat, giu nguyen OUT_DIR (trang thai luu moi 25 buoc)
NUM_VAL_FOR_SELECTION = 100     # dung toan bo val de chon best

from summ_rl import GroupRLConfig, RewardConfig
RL_CFG = GroupRLConfig(lr=1e-5, ppo_epochs=2, clip_eps=0.2, kl_coef=0.04, target_kl=0.05, adapt_kl=True,
                       sft_coef=0.1, temperature=0.9, min_new_tokens=48, use_amp=bf16, seed=42, **PROFILE)
REWARD_CFG = RewardConfig()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(42); np.random.seed(42); torch.manual_seed(42)
print('bf16:', bf16, '| device:', DEVICE, '| profile:', PROFILE)

In [ ]:
# Cell 3 - Mount Drive, doc du lieu, kiem tra
from google.colab import drive
drive.mount('/content/drive')

def load_split(name):
    path = Path(DATA_DIR) / name / 'dataset.jsonl'
    rows = [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]
    for r in rows:
        if not isinstance(r.get('source'), str) or not r['source'].strip() or not isinstance(r.get('summary'), str) or not r['summary'].strip():
            raise ValueError(f'Row {r.get("id")}: source/summary must be nonempty text')
    return rows

train_rows, val_rows, test_rows = load_split('train'), load_split('val'), load_split('test')
ids = [{r['id'] for r in rows} for rows in (train_rows, val_rows, test_rows)]
groups = [{r['group_id'] for r in rows} for rows in (train_rows, val_rows, test_rows)]
assert not (ids[0] & ids[1] or ids[0] & ids[2] or ids[1] & ids[2]), 'id overlap between splits'
assert not (groups[0] & groups[1] or groups[0] & groups[2] or groups[1] & groups[2]), 'duplicate group crosses splits'
val_sel = val_rows[:NUM_VAL_FOR_SELECTION]
print(len(train_rows), len(val_rows), len(test_rows), '| labels:', {r['label_status'] for r in train_rows})
assert Path(SFT_DIR).is_dir(), SFT_DIR
Path(OUT_DIR).mkdir(parents=True, exist_ok=RESUME)   # khong ghi de run cu (tru khi RESUME=True)

In [ ]:
# Cell 4 - Nap model: actor fp32 (master weights), reference dong bang
from transformers import AutoModelForSeq2SeqLM, T5Tokenizer
tokenizer = T5Tokenizer.from_pretrained(SFT_DIR, legacy=True)
model = AutoModelForSeq2SeqLM.from_pretrained(SFT_DIR, torch_dtype=torch.float32).to(DEVICE)
ref_model = AutoModelForSeq2SeqLM.from_pretrained(SFT_DIR, torch_dtype=torch.bfloat16 if bf16 else torch.float32).to(DEVICE)
print('actor params (M):', sum(p.numel() for p in model.parameters()) / 1e6, '| dtype:', next(model.parameters()).dtype)
if DEVICE.type == 'cuda':
    print('GPU memory used (GB):', round(torch.cuda.memory_allocated() / 1e9, 2))

In [ ]:
# Cell 5 - Kiem tra nhanh reward tren 1 mau (khong train)
from summ_rl import reward_breakdown
sample = val_rows[0]
print(reward_breakdown(sample['source'], sample['summary'], sample['summary'], REWARD_CFG))
print(reward_breakdown(sample['source'], sample['summary'].replace('0', '7'), sample['summary'], REWARD_CFG))

In [ ]:
# Cell 6 - Huan luyen RL. Dung som neu KL > kl_abort; best_model luon >= SFT baseline.
from summ_rl import GroupRLTrainer
trainer = GroupRLTrainer(model, ref_model, tokenizer, DEVICE, RL_CFG, REWARD_CFG)
history, best = trainer.train(train_rows, val_sel, steps=STEPS, out_dir=OUT_DIR, eval_every=EVAL_EVERY,
                              eval_kwargs=dict(num_beams=4, no_repeat_ngram_size=3, batch_size=8),
                              max_halluc_increase=1.0, save_every=25, resume=RESUME)
print('Best:', json.dumps(best, ensure_ascii=False, indent=2))

In [ ]:
# Cell 7 - Duong cong huan luyen
import pandas as pd
import matplotlib.pyplot as plt
df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 4, figsize=(18, 3.5))
panels = [(['reward'], 'Reward (sampled)'), (['kl', 'kl_coef'], 'KL to SFT'),
          (['hallucination', 'number_recall'], 'Number metrics'), (['active_groups', 'clip_fraction'], 'Signal / clipping')]
for ax, (cols, title) in zip(axes, panels):
    for c in cols:
        ax.plot(df['step'], df[c], label=c)
    ax.set_title(title); ax.legend()
plt.tight_layout(); plt.savefig(Path(OUT_DIR) / 'training_curves.png', dpi=130); plt.show()

In [ ]:
# Cell 8 - Danh gia SFT vs RL tren val day du; TEST chi chay mot lan sau khi chot model
from summ_rl.grpo import evaluate_model
eval_kwargs = dict(max_input_len=RL_CFG.max_input_len, max_new_tokens=RL_CFG.max_new_tokens, min_new_tokens=RL_CFG.min_new_tokens,
                   num_beams=4, no_repeat_ngram_size=3, batch_size=8, use_amp=bf16)
rl_model = AutoModelForSeq2SeqLM.from_pretrained(str(Path(OUT_DIR) / 'best_model'), torch_dtype=torch.float32).to(DEVICE)
table = {}
for name, m in [('SFT (Stage 2)', ref_model), ('RL v2', rl_model)]:
    table[name], _ = evaluate_model(m, tokenizer, val_rows, DEVICE, REWARD_CFG, **eval_kwargs)
print(pd.DataFrame(table).T.round(3).to_string())
(Path(OUT_DIR) / 'eval_val_sft_vs_rl.json').write_text(json.dumps(table, indent=2), encoding='utf-8')

In [ ]:
# Cell 9 - Test cuoi (chay MOT lan)
test_table = {}
for name, m in [('SFT (Stage 2)', ref_model), ('RL v2', rl_model)]:
    test_table[name], test_preds = evaluate_model(m, tokenizer, test_rows, DEVICE, REWARD_CFG, **eval_kwargs)
print(pd.DataFrame(test_table).T.round(3).to_string())
(Path(OUT_DIR) / 'eval_test_sft_vs_rl.json').write_text(json.dumps(test_table, indent=2), encoding='utf-8')

## Cách đọc kết quả

- Chỉ khẳng định RL tốt hơn khi `reward`, `rougeL` và `number_hallucination` đều không xấu đi so với SFT trên **val**, rồi mới xem **test**. Nhãn là pseudo-label, nên hãy đọc vài chục mẫu bằng mắt trước khi kết luận.
- `active_groups` thấp kéo dài nghĩa là các ứng viên giống nhau. Hãy tăng `temperature` lên 1.0 hoặc tăng `group_size`.
- `kl` vượt `kl_abort` thì giảm `lr` (5e-6) hoặc tăng `kl_coef`.
- Model lưu ở `OUT_DIR/best_model` dùng được trực tiếp trong notebook Gradio so sánh.